# Bayesian A/B Testing Engine — Analysis
End-to-end Bayesian analysis of a real 30-day Facebook marketing campaign A/B test
(control vs test creative). See `scripts/run_analysis.py` for the full reproducible pipeline;
this notebook walks through the same steps interactively.

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
from bayes_ab import (BetaBinomialModel, prob_b_beats_a, expected_loss, decision_summary,
    run_sequential, two_proportion_ztest, chi_square_test, prior_sensitivity,
    frequentist_sample_size)
from bayes_ab.visualization import plot_posteriors, plot_sequential_trajectory

## 1. Load the cleaned daily data
Each row = one campaign day: `trials` = impressions, `successes` = purchases.

In [ ]:
daily = pd.read_csv('../data/clean/ab_daily.csv')
daily.head(10)
print('days:', len(daily))
n_a, s_a = daily['trials_a'].sum(), daily['successes_a'].sum()
n_b, s_b = daily['trials_b'].sum(), daily['successes_b'].sum()
print(f'A: {s_a:,}/{n_a:,} = {s_a/n_a:.4%}')
print(f'B: {s_b:,}/{n_b:,} = {s_b/n_b:.4%}')

## 2. Fit the Beta-Binomial posteriors
Posterior = Beta(1 + s, 1 + n − s) under a uniform prior (see `docs/math_notes.md` for the derivation).

In [ ]:
a = BetaBinomialModel.fit(s_a, n_a)
b = BetaBinomialModel.fit(s_b, n_b)
prob = prob_b_beats_a(a, b)
print(f'P(B > A | data) = {prob:.6f}')
print(f'E[loss | choose B] = {expected_loss(a, b, "B"):.8f}')
print(f'E[loss | choose A] = {expected_loss(a, b, "A"):.8f}')
print('95% CI A:', a.credible_interval(), ' B:', b.credible_interval())
plot_posteriors(a, b, prob, 'posteriors_nb.png')

## 3. Sequential monitoring — how early could we have stopped?
Each day adds a batch of real data; the rule stops when P(B>A) ≥ 0.95 or the leader's expected loss is negligible.

In [ ]:
traj = run_sequential(daily)
traj[['day','rate_a','rate_b','prob_b_beats_a','expected_loss_b','decision']].tail(8)
plot_sequential_trajectory(traj, 'sequential_nb.png')

## 4. Frequentist cross-check (z-test, chi-square)

In [ ]:
z = two_proportion_ztest(s_a, n_a, s_b, n_b)
c = chi_square_test(s_a, n_a, s_b, n_b)
print(f"z = {z['z_statistic']:.2f}, p = {z['p_value']:.2e}")
print(f"95% CI for lift: [{z['ci_lower']:.5f}, {z['ci_upper']:.5f}]")
print(f"chi2 = {c['chi2_statistic']:.1f}, p = {c['p_value']:.2e}")

## 5. Prior sensitivity — do the conclusions survive different priors?

In [ ]:
sens = prior_sensitivity(s_a, n_a, s_b, n_b)
sens[['prior','prior_mean','prob_b_beats_a','expected_loss_choose_b']]

## 6. Sample-size planning for the next experiment

In [ ]:
print('Frequentist n/variant for 80% power:', frequentist_sample_size(0.005, 0.007))